<a href="https://colab.research.google.com/github/Kotyga/ML_toolkit/blob/main/%D0%94%D0%97_%D0%AF%D0%B7%D1%8B%D0%BA%D0%BE%D0%B2%D0%BE%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: символьное языковое моделирование

## Цель работы
Реализовать и сравнить две нейросетевые языковые модели, обученные генерировать текст на разных корпусах. Главный результат работы — **обоснованный вывод по экспериментам**, а не рекордное качество генерации.

## На чём сосредоточиться
- единый класс предобработки текста и преобразования символов в индексы и обратно;
- два источника текста: [**Александр Пушкин**](https://github.com/attatrol/data_sources/blob/master/onegin.txt) и [**Уильям Шекспир**](https://github.com/atsusy/ConvNetSharp/blob/ccaf6f6959c12026c4e1a0e396a7711654c92551/Examples/RnnDemo.Flow/shakespeare/sonnets.txt), а также **один собственный корпус**;
- две модели: простая RNN и LSTM;
- эксперименты с несколькими scheduler'ами;
- сравнение результатов и выводы по моделям, датасетам и параметрам.

## Ограничения
Не нужно добиваться идеальной генерации и проводить масштабный подбор гиперпараметров. Достаточно небольших, разумных экспериментов, которые позволяют сделать выводы. **Не подменяйте выводы впечатлениями: опирайтесь на loss, графики и примеры генерации.**

## Что должно быть в итоговой работе

1. Один класс, который загружает/принимает текст, выполняет предобработку, строит словари `token_to_idx` и `idx_to_token`, кодирует и декодирует текст, а также разделяет данные на train/test.
2. Три корпуса: текст Пушкина, текст Шекспира и собственный корпус (например, анекдоты, сказки, отзывы или другой подходящий текст).
3. Две модели, обучаемые по одной и той же задаче предсказания следующего символа.
4. Эксперименты как минимум с двумя scheduler'ами. Сравните поведение обучения на одинаковых или максимально близких условиях.
5. Таблица результатов и заключение: какая комбинация «модель + датасет + параметры» оказалась наиболее удачной и почему.

## 1. Импорты и воспроизводимость

Зафиксируйте seed, выберите устройство (`cpu` или `cuda`, если доступно) и подключите необходимые библиотеки.

In [ ]:
# your code here

## 2. Подготовка корпусов

Нужно использовать **все три источника**.

Для каждого источника запишите название, ссылку/происхождение, язык и приблизительный объём после очистки. Удалите технические заголовки, навигацию и явный мусор. Не удаляйте содержательные элементы только потому, что они кажутся необычными.

### Практический совет
Для сравнения моделей корпус не обязан быть огромным. Лучше использовать чистый, понятный корпус и явно описать ограничения, чем тратить время на сложный сбор данных.

In [ ]:
# Загрузите текст Пушкина.
# your code here

In [ ]:
# Загрузите текст Шекспира.
# your code here

In [ ]:
# Загрузите собственный корпус и укажите его источник.
# your code here

In [ ]:
# Краткая проверка: размеры, фрагменты текстов, пропуски/мусор.
# your code here

## 3. Единый класс предобработки и индексации

Реализуйте один класс, который можно использовать для каждого корпуса. Не создавайте три почти одинаковых класса.

Класс должен:
- принимать текст или идентификатор источника;
- нормализовать текст (например, привести к нижнему регистру; решение об удалении пунктуации обоснуйте);
- формировать множество символов и словари `token_to_idx` и `idx_to_token`;
- добавлять специальные символы начала/конца последовательности, если они используются;
- кодировать текст в индексы и декодировать индексы обратно;
- делить данные на обучающую и тестовую выборки без случайного перемешивания символов, разрушающего последовательность;
- обеспечивать одинаковый интерфейс для всех корпусов.

Продумайте, как обрабатывать символ, который не встречался в обучающей части. Не допускайте утечки тестовых данных в обучающую выборку при оценке качества.

In [ ]:
class TextPreprocessor:
    def __init__(self, text, lowercase=True, test_size=0.1):
        # your code here
        pass

    def preprocess(self, text):
        # your code here
        pass

    def build_vocabulary(self, train_text):
        # your code here
        pass

    def encode(self, text):
        # your code here
        pass

    def decode(self, indices):
        # your code here
        pass

    def split(self):
        # your code here
        pass

In [ ]:
# Создайте обработчики для трёх корпусов и проверьте кодирование/декодирование.
# your code here

### Проверка перед обучением
- [ ] Один и тот же класс работает с тремя корпусами.
- [ ] `decode(encode(text))` восстанавливает нормализованный текст (за исключением явно описанных специальных символов).
- [ ] Train и test не пересекаются по позициям исходного текста.
- [ ] Размер словаря и длина каждого корпуса выведены.

## 4. Подготовка обучающих примеров

Языковая модель предсказывает следующий символ по предыдущему контексту. Подготовьте входные последовательности и целевые последовательности со сдвигом на один символ.

In [ ]:
# Реализуйте создание последовательностей/батчей и DataLoader.
# Вход и target должны быть сдвинуты на один символ.
# your code here

## 5. Модель № 1 — простая RNN

Реализуйте символьную языковую модель на основе обычной рекуррентной сети. Модель должна получать входную последовательность и выдавать логиты для следующего символа на каждом шаге.

Можно использовать `Embedding`, `RNN`/`RNNCell` и линейный выходной слой. Укажите размеры embedding и hidden state.

In [ ]:
import torch
from torch import nn

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size=64, hidden_size=128):
        super().__init__()
        # your code here
        pass

    def forward(self, x, hidden=None):
        # your code here
        pass

## 6. Модель № 2 — LSTM

Реализуйте вторую модель на основе LSTM. Сохраните сопоставимыми основные настройки с RNN, насколько это возможно, чтобы сравнение было осмысленным.

Опишите, чем LSTM отличается от обычной RNN и почему её механизм памяти может помогать в моделировании длинных зависимостей.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_size=64, hidden_size=128):
        super().__init__()
        # your code here
        pass

    def forward(self, x, hidden=None):
        # your code here
        pass

## 7. Обучение и оценка

Реализуйте общую функцию обучения, пригодную для обеих моделей. Функция должна:
- обучать модель несколько эпох;
- считать train loss и validation/test loss;
- использовать `CrossEntropyLoss`;
- сохранять историю потерь;
- принимать оптимизатор и scheduler как аргументы;
- корректно переключать `model.train()` и `model.eval()`;
- использовать `torch.no_grad()` при оценке.

Если для выбора scheduler'а используется validation loss, выделите validation из обучающей части. Финальный test лучше оставить для итоговой оценки.

In [ ]:
def train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    scheduler,
    num_epochs,
    device
):
    history = {"train_loss": [], "validation_loss": []}
    # your code here
    return history

In [ ]:
# Реализуйте функцию построения графиков train/validation loss.
# your code here

## 8. Эксперименты с scheduler'ами

Сравните **как минимум два** scheduler'а, например:

- `ReduceLROnPlateau` — уменьшает learning rate, когда выбранная метрика перестаёт улучшаться;
- `StepLR` — уменьшает learning rate через заданное число эпох;
- `ExponentialLR` — постепенно уменьшает learning rate;
- `LambdaLR` — задаёт собственное расписание.

Выберите два варианта, обоснуйте параметры и сравнивайте их при одинаковых модели, корпусе, оптимизаторе и начальном learning rate. Для `ReduceLROnPlateau` передавайте scheduler'у именно ту метрику, по которой он должен принимать решение. Для остальных scheduler'ов соблюдайте соответствующий порядок вызова `step()`.

Не требуется перебрать всё. Достаточно выбрать несколько понятных комбинаций и объяснить наблюдаемое поведение.

In [ ]:
# Определите общий оптимизатор и первый scheduler.
# your code here

In [ ]:
# Проведите эксперимент с первым scheduler'ом и сохраните историю потерь.
# your code here

In [ ]:
# Проведите сравнимый эксперимент со вторым scheduler'ом.
# your code here

In [ ]:
# Постройте графики и сравните кривые обучения.
# your code here

### Честное сравнение
При сравнении scheduler'ов не меняйте одновременно корпус, архитектуру, batch size и learning rate: иначе будет непонятно, чем вызвано различие. Если параметры всё же отличаются, явно укажите это в отчёте.

## 9. Генерация текста

Реализуйте функцию генерации, которая:
1. принимает обученную модель и начальный фрагмент (*seed*);
2. предсказывает распределение следующего символа;
3. выбирает символ и добавляет его к контексту;
4. повторяет процесс до заданной длины.

Проверьте несколько значений температуры, например `0.2`, `0.5`, `1.0`. Используйте один и тот же начальный фрагмент для сравниваемых моделей одного корпуса.

### Как оценивать генерацию
Оценивайте не только «похожесть на оригинал». Обратите внимание на повторения, связность, характерные особенности корпуса и то, сохраняет ли модель структуру текста. Символьная модель может правдоподобно воспроизводить орфографию, но при этом генерировать бессмысленное содержание.

In [ ]:
def generate_text(model, preprocessor, seed, max_new_chars=300, temperature=1.0, device="cpu"):
    # your code here
    pass

In [ ]:
# Сгенерируйте примеры для обеих моделей на каждом из трёх корпусов.
# your code here

## 10. План экспериментов

Минимальный план:

- обучить RNN и LSTM на каждом из трёх корпусов;
- сравнить не менее двух scheduler'ов хотя бы на нескольких репрезентативных комбинациях;
- фиксировать train loss, validation loss, итоговую test loss и выбранные параметры;
- сгенерировать примеры текста с одинаковыми seed и настройками;
- не увеличивать вычислительные затраты без необходимости: цель — получить осмысленное сравнение, а не выжать максимальный результат.

Заполняйте таблицу по фактическим экспериментам. Не придумывайте результаты и не сравнивайте числа, полученные на несопоставимых данных.

In [ ]:
# Создайте таблицу результатов.
# Рекомендуемые столбцы:
# dataset, model, scheduler, learning_rate, batch_size, hidden_size,
# epochs, best_validation_loss, test_loss, temperature, observations
# your code here




### Что считать лучшим результатом
Не выбирайте победителя только по одному числу. Сопоставьте test loss с примерами генерации, устойчивостью обучения и размером/особенностями корпуса. Укажите, если метрики и визуальная оценка расходятся.

## 11. Выводы — главная часть работы

Напишите выводы своими словами. Ответьте как минимум на следующие вопросы:

1. Какая из двух моделей в целом оказалась лучше на каждом датасете? Всегда ли победила одна и та же архитектура?
2. На каком датасете модели обучались легче, а на каком хуже? Какие особенности корпуса могли на это повлиять?
3. Какой scheduler оказался удачнее и по каким признакам вы это определили? Видно ли изменение learning rate на графиках?
4. Какие значения параметров оказались разумными: learning rate, batch size, hidden size, число эпох и температура генерации?
5. Как изменялись сгенерированные тексты при изменении температуры?
6. Где модели повторяются, теряют структуру или генерируют бессмысленные фрагменты?
7. Какие ограничения эксперимента мешают делать сильные выводы (размер данных, число запусков, случайность и т. п.)?

### Критерий хорошего заключения
Хороший вывод связывает **наблюдение → свидетельство → объяснение**. Например: «На корпусе X у модели Y validation loss снижался стабильнее при scheduler Z; это видно на графике и согласуется с такими-то примерами генерации. Однако эксперимент ограничен одним запуском».

## 12. Чек-лист перед сдачей

- [ ] Использованы тексты Пушкина и Шекспира.
- [ ] Добавлен третий собственный корпус и указан его источник.
- [ ] Реализован единый класс предобработки и кодирования/декодирования.
- [ ] Обучены две модели: RNN и LSTM.
- [ ] Сравнены минимум два scheduler'а.
- [ ] Сохранены графики train и validation loss.
- [ ] Для моделей и датасетов приведены примеры генерации.
- [ ] Заполнена таблица результатов с параметрами и метриками.
- [ ] В выводах указано, какая комбинация модели, датасета и параметров оказалась лучшей и почему.
- [ ] Результаты не преувеличены: указаны ограничения и неудачные случаи.

## Дополнительные материалы

- [Karpathy — The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
- [PyTorch: Saving and Loading Models](https://pytorch.org/tutorials/beginner/saving_loading_models.html)
- [PyTorch: Learning Rate Schedulers](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)